# Build email extractor 

In [1]:
from pydantic import BaseModel, Field, EmailStr
from typing import Optional, Literal
from pydantic_ai import Agent
from constants import MODEL_LARGE
from dotenv import load_dotenv

load_dotenv()

class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_category: Literal["damadged product", "billing", "technical", "other"]
    urgency: Literal["low", "medium", "high"]
    summary: str = Field(description="summarize the email in 3-4 sentences.")

email_extractor_agent = Agent(MODEL_LARGE, system_prompt="""
You are customer support agent, your task is to 
extract relevant information from an email.
""", output_type= EmailExtractor)


TODO: Load in the data and test the agent on one email

In [6]:
import pandas as pd

df = pd.read_json("data/emails_cleaned.json")
df

,inputs,expectations
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'expected_response': '{""sender_name"": ""Erik L..."
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'expected_response': '{""sender_name"": ""Maja B..."
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'expected_response': '{""sender_name"": ""Linnea..."


In [7]:
sample_mail = df.iloc[2]["inputs"]["email"]
sample_mail

"From: Oscar Johansson <oscar.johansson@yahoo.se>\nSubject: Cannot access my account for 3 days - Urgent help needed\n\nHello Support Team,\n\nI am reaching out because I have been completely locked out of my account for the past three days and I am running out of ideas on how to fix this on my own. The problem started on Monday evening when I tried to log in as usual but kept receiving an 'Invalid credentials' error despite being absolutely certain that I was entering the correct password.\n\nI followed the instructions on your website to reset my password, but the problem is that the password reset email never arrives in my inbox. I have checked my spam and junk folders multiple times, and there is nothing there either. I have attempted the reset process at least six or seven times across different browsers and even from my phone, but the result is always the same - no email arrives.\n\nThis is causing me real problems because I have important documents and data stored in my account 

In [15]:
result = await email_extractor_agent.run(sample_mail)

result.output

EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary='Oscar Johanson cannot access his account for three days, receiving invalid credentials errors. Password reset emails never arrive despite multiple attempts and checks of spam folders. He needs urgent access to important work documents before a Friday deadline and is willing to verify his identity.')

TODO: show ground truth and compare

## Load in prompts from mlflow

In [ ]:
from mlflow.genai import load_prompt


class EmailExtractor(BaseModel):
    sender_name: Optional[str]
    sender_email: EmailStr
    issue_category: Literal["damadged product", "billing", "technical", "other"]
    urgency: Literal["low", "medium", "high"] = Field(
        description=load_prompt("email-urgency-description").format()
    )
    summary: str = Field(
        description=load_prompt("summary-description").format(num_sentences=4)
    )


email_extractor_agent = Agent(
    MODEL_LARGE,
    system_prompt=load_prompt("email-extractor-system-prompt").format(),
    output_type=EmailExtractor,
)

In [8]:
result = await email_extractor_agent.run(sample_mail)
result.output

EmailExtractor(sender_name='Oscar Johansson', sender_email='oscar.johansson@yahoo.se', issue_category='technical', urgency='high', summary='Oscar Johansson cannot access his account for three days due to invalid credentials and not receiving password reset emails despite multiple attempts and checks. He has tried resetting the password, checking spam, using different browsers and devices, and clearing cache, all without success. He urgently needs access to important work documents before a Friday deadline and is willing to verify his identity. He requests immediate assistance to resolve the issue.')

## LLM Judge

1. require data with columns: inputs, expectations, outputs
2. mlflow experiments

In [9]:
result.output.model_dump()

{'sender_name': 'Oscar Johansson',
 'sender_email': 'oscar.johansson@yahoo.se',
 'issue_category': 'technical',
 'urgency': 'high',
 'summary': 'Oscar Johansson cannot access his account for three days due to invalid credentials and not receiving password reset emails despite multiple attempts and checks. He has tried resetting the password, checking spam, using different browsers and devices, and clearing cache, all without success. He urgently needs access to important work documents before a Friday deadline and is willing to verify his identity. He requests immediate assistance to resolve the issue.'}

TODO: dataframe with inputs, expectations, outputs but only 1 row

In [13]:
df["outputs"] = [{},{}, result.output.model_dump(), {}]
df

,inputs,expectations,outputs
0,{'email': 'From: Erik Lindqvist <erik.lindqvis...,"{'expected_response': '{""sender_name"": ""Erik L...",{}
1,{'email': 'From: Maja Bergström <maja.bergstro...,"{'expected_response': '{""sender_name"": ""Maja B...",{}
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ...","{'sender_name': 'Oscar Johansson', 'sender_ema..."
3,{'email': 'From: Linnea Karlsson <linnea.karls...,"{'expected_response': '{""sender_name"": ""Linnea...",{}


In [14]:
df_sample = df.drop([0,1,3])
df_sample

,inputs,expectations,outputs
2,{'email': 'From: Oscar Johansson <oscar.johans...,"{'expected_response': '{""sender_name"": ""Oscar ...","{'sender_name': 'Oscar Johansson', 'sender_ema..."


# LLM judge

In [16]:
from mlflow.genai.scorers import get_all_scorers


In [17]:
from mlflow.genai.scorers import Correctness, Summarization, Completeness, Fluency
import mlflow

llm_judge = "openrouter:/nvidia/nemotron-3-nano-30b-a3b:free"

with mlflow.start_run(run_name="email-extractor-avaluation"):
    mlflow.log_param("model_judge", llm_judge)
    mlflow.log_param("model", MODEL_LARGE)

    results = mlflow.genai.evaluate(
        data = df_sample,
        scorers=[
            Correctness(model=llm_judge),
            Summarization(model=llm_judge)
        ]
    )

results

2026/04/15 14:54:09 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet

2026/04/15 14:54:10 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled durin


✨ Evaluation completed.

Metrics and evaluation results are logged to the MLflow run:
  Run name: email-extractor-avaluation
  Run ID: 73f6ed11245f4c4083e4aa3b7a4e379f

To view the detailed evaluation results with sample-wise scores,
open the Traces tab in the Run page in the MLflow UI.



EvaluationResult(
  run_id: 73f6ed11245f4c4083e4aa3b7a4e379f
  metrics:
    summarization/mean: 1.0
  result_df: 1 rows x 15 cols
)

In [18]:
results.result_df

,trace_id,summarization/value,correctness/value,expected_response/value,trace,client_request_id,state,request_time,execution_duration,request,response,trace_metadata,tags,spans,assessments
0,tr-eb627fbe0ed96b5e0ecefd4a88a07ac0,yes,None,"{""sender_name"": ""Oscar Johansson"", ""sender_ema...","{""info"": {""trace_id"": ""tr-eb627fbe0ed96b5e0ece...",None,OK,1776257654573,0,{'email': 'From: Oscar Johansson <oscar.johans...,"{'sender_name': 'Oscar Johansson', 'sender_ema...","{'mlflow.user': 'norel', 'mlflow.source.name':...",{'mlflow.artifactLocation': 'file:c:/Users/nor...,"[{'trace_id': '62J/vg7Za14Ozv1KiKB6wA==', 'spa...",[{'assessment_id': 'a-1ad4b368e8e44d298e030e20...
